In [1]:
import os
import pandas as pd
import numpy as np

os.chdir("C:/Users/Lenovo/Desktop/Agai Project")  # adjust if needed
print("Working dir:", os.getcwd())

FEATURE_COLS = [f"mfcc_{i}" for i in range(13)] + ["pitch_mean", "energy_mean", "zcr_mean", "duration"]
TARGET_COL = "confidence_label"

train_df = pd.read_csv("data/processed/audio/train.csv")
val_df = pd.read_csv("data/processed/audio/val.csv")
test_df = pd.read_csv("data/processed/audio/test.csv")

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print(train_df[TARGET_COL].value_counts())

Working dir: C:\Users\Lenovo\Desktop\Agai Project
Train: (1008, 21) Val: (216, 21) Test: (216, 21)
confidence_label
nervous      538
confident    336
neutral      134
Name: count, dtype: int64


In [2]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
X_train, y_train = train_df[FEATURE_COLS], label_encoder.fit_transform(train_df[TARGET_COL])
X_val, y_val = val_df[FEATURE_COLS], label_encoder.transform(val_df[TARGET_COL])
X_test, y_test = test_df[FEATURE_COLS], label_encoder.transform(test_df[TARGET_COL])

print("Classes:", list(label_encoder.classes_))

Classes: ['confident', 'nervous', 'neutral']


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

baseline_clf = RandomForestClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=2,
    class_weight="balanced", random_state=42, n_jobs=-1
)
baseline_clf.fit(X_train, y_train)

print("BASELINE (original)")
print("Train acc:", accuracy_score(y_train, baseline_clf.predict(X_train)))
print("Val acc:  ", accuracy_score(y_val, baseline_clf.predict(X_val)))
print("Test acc: ", accuracy_score(y_test, baseline_clf.predict(X_test)))

BASELINE (original)
Train acc: 1.0
Val acc:   0.6805555555555556
Test acc:  0.7222222222222222


In [4]:
capped_clf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=5,
    class_weight="balanced", random_state=42, n_jobs=-1
)
capped_clf.fit(X_train, y_train)

print("CAPPED DEPTH (max_depth=10, min_samples_leaf=5)")
print("Train acc:", accuracy_score(y_train, capped_clf.predict(X_train)))
print("Val acc:  ", accuracy_score(y_val, capped_clf.predict(X_val)))
print("Test acc: ", accuracy_score(y_test, capped_clf.predict(X_test)))

CAPPED DEPTH (max_depth=10, min_samples_leaf=5)
Train acc: 0.9345238095238095
Val acc:   0.6620370370370371
Test acc:  0.6898148148148148


In [5]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(capped_clf, X_train, y_train, cv=5, scoring='accuracy')
print("5-fold CV accuracy per fold:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean(), "| Std:", cv_scores.std())

5-fold CV accuracy per fold: [0.67326733 0.72277228 0.74752475 0.66169154 0.68656716]
Mean CV accuracy: 0.698364612580661 | Std: 0.03202005958255332


In [6]:
import sys
!{sys.executable} -m pip install xgboost

from xgboost import XGBClassifier

xgb_clf = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    random_state=42, eval_metric='mlogloss'
)
xgb_clf.fit(X_train, y_train)

print("XGBOOST")
print("Train acc:", accuracy_score(y_train, xgb_clf.predict(X_train)))
print("Val acc:  ", accuracy_score(y_val, xgb_clf.predict(X_val)))
print("Test acc: ", accuracy_score(y_test, xgb_clf.predict(X_test)))


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


XGBOOST
Train acc: 1.0
Val acc:   0.7037037037037037
Test acc:  0.7546296296296297


In [7]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2]
}

grid_search = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='mlogloss'),
    param_grid, cv=3, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
best_xgb = grid_search.best_estimator_

print("\nBEST XGBOOST (tuned)")
print("Train acc:", accuracy_score(y_train, best_xgb.predict(X_train)))
print("Val acc:  ", accuracy_score(y_val, best_xgb.predict(X_val)))
print("Test acc: ", accuracy_score(y_test, best_xgb.predict(X_test)))

Best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}

BEST XGBOOST (tuned)
Train acc: 1.0
Val acc:   0.6990740740740741
Test acc:  0.7592592592592593


In [8]:
# Filter out 'neutral' class entirely — merge into simpler, more useful binary problem
train_binary = train_df[train_df[TARGET_COL] != 'neutral'].copy()
val_binary = val_df[val_df[TARGET_COL] != 'neutral'].copy()
test_binary = test_df[test_df[TARGET_COL] != 'neutral'].copy()

le_binary = LabelEncoder()
Xb_train, yb_train = train_binary[FEATURE_COLS], le_binary.fit_transform(train_binary[TARGET_COL])
Xb_val, yb_val = val_binary[FEATURE_COLS], le_binary.transform(val_binary[TARGET_COL])
Xb_test, yb_test = test_binary[FEATURE_COLS], le_binary.transform(test_binary[TARGET_COL])

print("Binary classes:", list(le_binary.classes_))
print("Train size:", len(train_binary), "Val size:", len(val_binary), "Test size:", len(test_binary))

binary_xgb = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    random_state=42, eval_metric='logloss'
)
binary_xgb.fit(Xb_train, yb_train)

print("\nBINARY (confident vs nervous only)")
print("Train acc:", accuracy_score(yb_train, binary_xgb.predict(Xb_train)))
print("Val acc:  ", accuracy_score(yb_val, binary_xgb.predict(Xb_val)))
print("Test acc: ", accuracy_score(yb_test, binary_xgb.predict(Xb_test)))

Binary classes: ['confident', 'nervous']
Train size: 874 Val size: 187 Test size: 187

BINARY (confident vs nervous only)
Train acc: 1.0
Val acc:   0.7647058823529411
Test acc:  0.7914438502673797


In [9]:
from sklearn.metrics import classification_report, confusion_matrix

results_summary = pd.DataFrame({
    "Model": ["Baseline RF", "Capped RF", "XGBoost", "Tuned XGBoost", "Binary XGBoost"],
    "Val Accuracy": [
        accuracy_score(y_val, baseline_clf.predict(X_val)),
        accuracy_score(y_val, capped_clf.predict(X_val)),
        accuracy_score(y_val, xgb_clf.predict(X_val)),
        accuracy_score(y_val, best_xgb.predict(X_val)),
        accuracy_score(yb_val, binary_xgb.predict(Xb_val)),
    ],
    "Test Accuracy": [
        accuracy_score(y_test, baseline_clf.predict(X_test)),
        accuracy_score(y_test, capped_clf.predict(X_test)),
        accuracy_score(y_test, xgb_clf.predict(X_test)),
        accuracy_score(y_test, best_xgb.predict(X_test)),
        accuracy_score(yb_test, binary_xgb.predict(Xb_test)),
    ]
})
print(results_summary)

            Model  Val Accuracy  Test Accuracy
0     Baseline RF      0.680556       0.722222
1       Capped RF      0.662037       0.689815
2         XGBoost      0.703704       0.754630
3   Tuned XGBoost      0.699074       0.759259
4  Binary XGBoost      0.764706       0.791444


In [10]:
# CHANGE THIS to whichever model won in Cell 9
winning_model = best_xgb   # or binary_xgb, capped_clf, etc.
winning_X_test = X_test     # or Xb_test if binary won
winning_y_test = y_test     # or yb_test if binary won
winning_labels = label_encoder.classes_   # or le_binary.classes_ if binary won

y_pred = winning_model.predict(winning_X_test)
print(classification_report(winning_y_test, y_pred, target_names=winning_labels, zero_division=0))

cm = confusion_matrix(winning_y_test, y_pred)
print(pd.DataFrame(cm, index=winning_labels, columns=winning_labels))

              precision    recall  f1-score   support

   confident       0.72      0.71      0.71        72
     nervous       0.79      0.86      0.82       115
     neutral       0.70      0.48      0.57        29

    accuracy                           0.76       216
   macro avg       0.74      0.68      0.70       216
weighted avg       0.76      0.76      0.75       216

           confident  nervous  neutral
confident         51       16        5
nervous           15       99        1
neutral            5       10       14


In [2]:
import os
os.chdir("C:/Users/Lenovo/Desktop/Agai Project")  # your actual project root
print(os.getcwd())

C:\Users\Lenovo\Desktop\Agai Project


In [3]:
print(os.path.exists("data/processed/audio/train.csv"))

True


In [4]:
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

FEATURE_COLS = [f"mfcc_{i}" for i in range(13)] + ["pitch_mean", "energy_mean", "zcr_mean", "duration"]
TARGET_COL = "confidence_label"

train_df = pd.read_csv("data/processed/audio/train.csv")
val_df = pd.read_csv("data/processed/audio/val.csv")
test_df = pd.read_csv("data/processed/audio/test.csv")

# Drop 'neutral' — binary confident vs nervous only
train_binary = train_df[train_df[TARGET_COL] != 'neutral'].copy()
val_binary = val_df[val_df[TARGET_COL] != 'neutral'].copy()
test_binary = test_df[test_df[TARGET_COL] != 'neutral'].copy()

le_binary = LabelEncoder()
Xb_train, yb_train = train_binary[FEATURE_COLS], le_binary.fit_transform(train_binary[TARGET_COL])
Xb_val, yb_val = val_binary[FEATURE_COLS], le_binary.transform(val_binary[TARGET_COL])
Xb_test, yb_test = test_binary[FEATURE_COLS], le_binary.transform(test_binary[TARGET_COL])

binary_xgb = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    random_state=42, eval_metric='logloss'
)
binary_xgb.fit(Xb_train, yb_train)

print("Val acc:", accuracy_score(yb_val, binary_xgb.predict(Xb_val)))
print("Test acc:", accuracy_score(yb_test, binary_xgb.predict(Xb_test)))

# Save as v2 — this is what confidence_agent.py now expects
joblib.dump(binary_xgb, "models/confidence_classifier_v2.joblib")
joblib.dump(le_binary, "models/confidence_label_encoder_v2.joblib")
print("\nSaved: models/confidence_classifier_v2.joblib")
print("Saved: models/confidence_label_encoder_v2.joblib")

Val acc: 0.7647058823529411
Test acc: 0.7914438502673797

Saved: models/confidence_classifier_v2.joblib
Saved: models/confidence_label_encoder_v2.joblib
